# This Notebook is to run STREME/MEME, TOMTOM and FIMO

## Set UP

In [2]:
from pathlib import Path
import subprocess
import pandas as pd
import os
from datetime import datetime
import urllib.request

In [3]:
PROJECT = Path("/s/project/ml4rg_students/2026/project15")

FASTA_DIR = PROJECT / "working" / "sequence_datasets_fastas"
RESULT_DIR = PROJECT / "working" / "streme_results"
LOG_DIR = PROJECT / "working" / "logs" / "streme_notebook"

MEME_ENV = Path("/opt/modules/i12g/anaconda/envs/meme_env")
MEME_BIN = MEME_ENV / "bin"

STREME = MEME_BIN / "streme"
TOMTOM = MEME_BIN / "tomtom"
FIMO = MEME_BIN / "fimo"

JASPAR_DIR = PROJECT / "working" / "jaspar"
JASPAR_DIR.mkdir(parents=True, exist_ok=True)

JASPAR_FUNGI = JASPAR_DIR / "JASPAR2026_CORE_fungi_non-redundant_pfms_meme.txt"

url = "https://jaspar.elixir.no/download/data/2026/CORE/JASPAR2026_CORE_fungi_non-redundant_pfms_meme.txt"

if not JASPAR_FUNGI.exists() or JASPAR_FUNGI.stat().st_size == 0:
    urllib.request.urlretrieve(url, JASPAR_FUNGI)

print(JASPAR_FUNGI)
print("exists:", JASPAR_FUNGI.exists())
print("size:", JASPAR_FUNGI.stat().st_size)

for d in [
    RESULT_DIR / "streme",
    RESULT_DIR / "tomtom",
    RESULT_DIR / "fimo_jaspar",
    RESULT_DIR / "fimo_streme",
    LOG_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

print("FASTA_DIR:", FASTA_DIR)
print("RESULT_DIR:", RESULT_DIR)
print("JASPAR:", JASPAR_DIR)

/s/project/ml4rg_students/2026/project15/working/jaspar/JASPAR2026_CORE_fungi_non-redundant_pfms_meme.txt
exists: True
size: 87218
FASTA_DIR: /s/project/ml4rg_students/2026/project15/working/sequence_datasets_fastas
RESULT_DIR: /s/project/ml4rg_students/2026/project15/working/streme_results
JASPAR: /s/project/ml4rg_students/2026/project15/working/jaspar


## Preperation

In [4]:
for tool in [STREME, TOMTOM, FIMO, JASPAR]:
    print(tool, "exists:", tool.exists())

NameError: name 'JASPAR' is not defined

In [5]:
fastas = sorted(list(FASTA_DIR.glob("*.fa")) + list(FASTA_DIR.glob("*.fasta")))

print("Number of FASTA files:", len(fastas))
for f in fastas[:10]:
    print(f.name)

Number of FASTA files: 1401
_candida_arabinofermentans_nrrl_yb_2248_gca_001661425_sequence_mapper.fasta
_candida_auris_gca_001189475_sequence_mapper.fasta
_candida_auris_gca_002775015_sequence_mapper.fasta
_candida_auris_gca_003013715_sequence_mapper.fasta
_candida_auris_gca_003014415_sequence_mapper.fasta
_candida_auris_gca_007168705_sequence_mapper.fasta
_candida_auris_gca_008275145_sequence_mapper.fasta
_candida_glabrata_gca_001466525_sequence_mapper.fasta
_candida_glabrata_gca_001466535_sequence_mapper.fasta
_candida_glabrata_gca_001466565_sequence_mapper.fasta


In [6]:
def fasta_name(fasta_path: Path) -> str:
    name = fasta_path.name
    if name.endswith(".fasta"):
        name = name[:-6]
    elif name.endswith(".fa"):
        name = name[:-3]
    return name


def paths_for_fasta(fasta_path: Path):
    name = fasta_name(fasta_path)
    
    streme_out = RESULT_DIR / "streme" / name
    tomtom_out = RESULT_DIR / "tomtom" / name
    fimo_jaspar_out = RESULT_DIR / "fimo_jaspar" / name
    fimo_streme_out = RESULT_DIR / "fimo_streme" / name
    
    return {
        "name": name,
        "streme_out": streme_out,
        "tomtom_out": tomtom_out,
        "fimo_jaspar_out": fimo_jaspar_out,
        "fimo_streme_out": fimo_streme_out,
        "streme_txt": streme_out / "streme.txt",
        "tomtom_tsv": tomtom_out / "tomtom.tsv",
        "fimo_jaspar_tsv": fimo_jaspar_out / "fimo.tsv",
        "fimo_streme_tsv": fimo_streme_out / "fimo.tsv",
    }


def exists_nonempty(path: Path) -> bool:
    return path.exists() and path.stat().st_size > 0


def run_command(cmd, log_prefix: Path):
    """
    Runs a command and writes stdout/stderr to log files.
    Raises an error if the command fails.
    """
    log_prefix.parent.mkdir(parents=True, exist_ok=True)
    
    stdout_file = log_prefix.with_suffix(".out")
    stderr_file = log_prefix.with_suffix(".err")
    
    print("Running:")
    print(" ".join(map(str, cmd)))
    print("stdout:", stdout_file)
    print("stderr:", stderr_file)
    
    with open(stdout_file, "w") as out, open(stderr_file, "w") as err:
        result = subprocess.run(
            list(map(str, cmd)),
            stdout=out,
            stderr=err,
            text=True,
        )
    
    if result.returncode != 0:
        raise RuntimeError(
            f"Command failed with exit code {result.returncode}. Check {stderr_file}"
        )

### Check status


In [7]:
rows = []

for fasta in fastas:
    p = paths_for_fasta(fasta)
    rows.append({
        "name": p["name"],
        "fasta": str(fasta),
        "streme": exists_nonempty(p["streme_txt"]),
        "tomtom": exists_nonempty(p["tomtom_tsv"]),
        "fimo_jaspar": exists_nonempty(p["fimo_jaspar_tsv"]),
        "fimo_streme": exists_nonempty(p["fimo_streme_tsv"]),
    })

status = pd.DataFrame(rows)
status

,name,fasta,streme,tomtom,fimo_jaspar,fimo_streme
0,_candida_arabinofermentans_nrrl_yb_2248_gca_00...,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,True
1,_candida_auris_gca_001189475_sequence_mapper,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,True
2,_candida_auris_gca_002775015_sequence_mapper,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,True
3,_candida_auris_gca_003013715_sequence_mapper,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,True
4,_candida_auris_gca_003014415_sequence_mapper,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,True
...,...,...,...,...,...,...
1396,zygotorulaspora_mrakii_gca_013402915_sequence_...,/s/project/ml4rg_students/2026/project15/worki...,False,False,False,False
1397,zymoseptoria_brevis_gca_000966595_sequence_mapper,/s/project/ml4rg_students/2026/project15/worki...,False,False,False,False
1398,zymoseptoria_tritici_sequence_mapper,/s/project/ml4rg_students/2026/project15/worki...,False,False,False,False
1399,zymoseptoria_tritici_st99ch_1a5_gca_900099495_...,/s/project/ml4rg_students/2026/project15/worki...,False,False,False,False


In [8]:
status[["streme", "tomtom", "fimo_jaspar", "fimo_streme"]].sum()

streme         12
tomtom         12
fimo_jaspar    11
fimo_streme    11
dtype: int64

## Running Pipeline

In [21]:
def run_pipeline_for_fasta(
    fasta: Path,
    force: bool = False,
    streme_time: int = 1800,
    minw: int = 6,
    maxw: int = 20,
    nmotifs: int = 10,
    fimo_thresh: str = "1e-4",
    fimo_max_stored_scores: int = 100000,
    fimo_skip_matched_sequence: bool = True,
):
    p = paths_for_fasta(fasta)
    name = p["name"]

    safe_print("=" * 80)
    safe_print(f"Processing: {name}")
    safe_print(f"FASTA: {fasta}")
    safe_print("=" * 80)

    for key in ["streme_out", "tomtom_out", "fimo_jaspar_out", "fimo_streme_out"]:
        p[key].mkdir(parents=True, exist_ok=True)

    # -------------------------
    # STREME
    # -------------------------
    streme_cmd = [
        STREME,
        "--dna",
        "--p", fasta,
        "--oc", p["streme_out"],
        "--minw", minw,
        "--maxw", maxw,
        "--nmotifs", nmotifs,
        "--time", streme_time,
        "--verbosity", 1,
    ]

    run_step_if_needed(
        sample_name=name,
        label="STREME",
        expected_output=p["streme_txt"],
        cmd=streme_cmd,
        log_file=LOG_DIR / f"{name}.streme",
        force=force,
    )

    # -------------------------
    # FIMO with JASPAR motifs
    # unabhängig von STREME
    # -------------------------
    fimo_jaspar_cmd = [
        FIMO,
        "--oc", p["fimo_jaspar_out"],
        "--thresh", fimo_thresh,
        "--max-stored-scores", fimo_max_stored_scores,
    ]

    if fimo_skip_matched_sequence:
        fimo_jaspar_cmd.append("--skip-matched-sequence")

    fimo_jaspar_cmd += [
        JASPAR_FUNGI,
        fasta,
    ]

    run_step_if_needed(
        sample_name=name,
        label="FIMO JASPAR",
        expected_output=p["fimo_jaspar_tsv"],
        cmd=fimo_jaspar_cmd,
        log_file=LOG_DIR / f"{name}.fimo_jaspar",
        force=force,
    )

    # Ab hier brauchen wir STREME-Ergebnisse
    if not exists_nonempty(p["streme_txt"]):
        safe_print(f"[{name}] STREME result missing. Skipping TOMTOM and FIMO_STREME.")
        return

    # -------------------------
    # TOMTOM
    # -------------------------
    tomtom_cmd = [
        TOMTOM,
        "-oc", p["tomtom_out"],
        "-verbosity", 1,
        "-min-overlap", 5,
        "-dist", "pearson",
        "-evalue",
        "-thresh", 10,
        p["streme_txt"],
        JASPAR_FUNGI,
    ]

    run_step_if_needed(
        sample_name=name,
        label="TOMTOM",
        expected_output=p["tomtom_tsv"],
        cmd=tomtom_cmd,
        log_file=LOG_DIR / f"{name}.tomtom",
        force=force,
    )

    # -------------------------
    # FIMO with STREME motifs
    # -------------------------
    fimo_streme_cmd = [
        FIMO,
        "--oc", p["fimo_streme_out"],
        "--thresh", fimo_thresh,
        "--max-stored-scores", fimo_max_stored_scores,
    ]

    if fimo_skip_matched_sequence:
        fimo_streme_cmd.append("--skip-matched-sequence")

    fimo_streme_cmd += [
        p["streme_txt"],
        fasta,
    ]

    run_step_if_needed(
        sample_name=name,
        label="FIMO STREME",
        expected_output=p["fimo_streme_tsv"],
        cmd=fimo_streme_cmd,
        log_file=LOG_DIR / f"{name}.fimo_streme",
        force=force,
    )

    safe_print(f"[{name}] Finished.")

### To test just for one FASTA

In [10]:
test_fasta = fastas[0]
test_fasta

PosixPath('/s/project/ml4rg_students/2026/project15/working/sequence_datasets_fastas/_candida_arabinofermentans_nrrl_yb_2248_gca_001661425_sequence_mapper.fasta')

In [29]:
run_pipeline_for_fasta(
    test_fasta,
    force=False,
    streme_time=1800,
    minw=6,
    maxw=30,
    nmotifs=20,
    fimo_thresh="1e-4",
)

Processing: _candida_arabinofermentans_nrrl_yb_2248_gca_001661425_sequence_mapper
FASTA: /s/project/ml4rg_students/2026/project15/working/sequence_datasets_fastas/_candida_arabinofermentans_nrrl_yb_2248_gca_001661425_sequence_mapper.fasta
STREME exists, skipping.
TOMTOM exists, skipping.
Running FIMO with JASPAR.
Running:
/opt/modules/i12g/anaconda/envs/meme_env/bin/fimo --oc /s/project/ml4rg_students/2026/project15/working/streme_results/fimo_jaspar/_candida_arabinofermentans_nrrl_yb_2248_gca_001661425_sequence_mapper --thresh 1e-4 --max-stored-scores 1000000 /s/project/ml4rg_students/2026/project15/working/jaspar/JASPAR2026_CORE_fungi_non-redundant_pfms_meme.txt /s/project/ml4rg_students/2026/project15/working/sequence_datasets_fastas/_candida_arabinofermentans_nrrl_yb_2248_gca_001661425_sequence_mapper.fasta
stdout: /s/project/ml4rg_students/2026/project15/working/logs/streme_notebook/_candida_arabinofermentans_nrrl_yb_2248_gca_001661425_sequence_mapper.out
stderr: /s/project/ml4rg_

### For all FASTAS

In [18]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import subprocess
import threading
import os

In [22]:
PRINT_LOCK = threading.Lock()


def safe_print(*args, **kwargs):
    with PRINT_LOCK:
        print(*args, **kwargs, flush=True)


def run_command(cmd, log_prefix: Path):
    log_prefix = Path(log_prefix)
    log_prefix.parent.mkdir(parents=True, exist_ok=True)

    # Wichtig:
    # NICHT log_prefix.with_suffix(".out") verwenden,
    # sonst wird aus "sample.streme" nur "sample.out".
    stdout_file = log_prefix.parent / f"{log_prefix.name}.out"
    stderr_file = log_prefix.parent / f"{log_prefix.name}.err"

    cmd = [str(x) for x in cmd]

    safe_print("Running:")
    safe_print(" ".join(cmd))
    safe_print(f"stdout: {stdout_file}")
    safe_print(f"stderr: {stderr_file}")

    with open(stdout_file, "w") as out, open(stderr_file, "w") as err:
        subprocess.run(
            cmd,
            stdout=out,
            stderr=err,
            check=True,
        )

In [24]:
def run_step_if_needed(
    sample_name: str,
    label: str,
    expected_output: Path,
    cmd: list,
    log_file: Path,
    force: bool = False,
):
    if exists_nonempty(expected_output) and not force:
        #safe_print(f"[{sample_name}] {label} exists, skipping.")
        return

    #safe_print(f"[{sample_name}] Running {label}.")
    run_command(cmd, log_file)

In [25]:
def run_all_fastas(
    fastas,
    force: bool = False,
    streme_time: int = 1800,
    minw: int = 6,
    maxw: int = 20,
    nmotifs: int = 10,
    fimo_thresh: str = "1e-4",
    max_workers: int | None = None,
):
    fastas = list(fastas)

    if max_workers is None:
        max_workers = min(len(fastas), max(1, (os.cpu_count() or 2) // 2))

    safe_print(f"Running {len(fastas)} FASTA files with max_workers={max_workers}")

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {}

        for fasta in fastas:
            fut = executor.submit(
                run_pipeline_for_fasta,
                fasta,
                force=force,
                streme_time=streme_time,
                minw=minw,
                maxw=maxw,
                nmotifs=nmotifs,
                fimo_thresh=fimo_thresh,
            )
            futures[fut] = fasta

        for i, fut in enumerate(as_completed(futures), start=1):
            fasta = futures[fut]

            try:
                fut.result()
                safe_print(f"\n### completed {i}/{len(fastas)}: {fasta.name} ###")
            except Exception as e:
                safe_print(f"\n### FAILED {i}/{len(fastas)}: {fasta.name} ###")
                safe_print(f"ERROR for {fasta.name}: {e}")
                safe_print("Continuing with next FASTA.")

In [ ]:
run_all_fastas(
    fastas,
    force=False,
    streme_time=1800,
    minw=6,
    maxw=20,
    nmotifs=10,
    fimo_thresh="1e-4",
    max_workers=4,
)

Running 1401 FASTA files with max_workers=4
Processing: _candida_arabinofermentans_nrrl_yb_2248_gca_001661425_sequence_mapper
FASTA: /s/project/ml4rg_students/2026/project15/working/sequence_datasets_fastas/_candida_arabinofermentans_nrrl_yb_2248_gca_001661425_sequence_mapper.fasta
Processing: _candida_auris_gca_001189475_sequence_mapper
Processing: _candida_auris_gca_003013715_sequence_mapper
FASTA: /s/project/ml4rg_students/2026/project15/working/sequence_datasets_fastas/_candida_auris_gca_001189475_sequence_mapper.fasta
Processing: _candida_auris_gca_002775015_sequence_mapper
FASTA: /s/project/ml4rg_students/2026/project15/working/sequence_datasets_fastas/_candida_auris_gca_002775015_sequence_mapper.fasta
FASTA: /s/project/ml4rg_students/2026/project15/working/sequence_datasets_fastas/_candida_auris_gca_003013715_sequence_mapper.fasta
[_candida_arabinofermentans_nrrl_yb_2248_gca_001661425_sequence_mapper] Finished.
Processing: _candida_auris_gca_003014415_sequence_mapper
FASTA: /s/p

### Update status

In [1]:
rows = []

for fasta in fastas:
    p = paths_for_fasta(fasta)
    rows.append({
        "name": p["name"],
        "streme": exists_nonempty(p["streme_txt"]),
        "tomtom": exists_nonempty(p["tomtom_tsv"]),
        "fimo_jaspar": exists_nonempty(p["fimo_jaspar_tsv"]),
        "fimo_streme": exists_nonempty(p["fimo_streme_tsv"]),
    })

status = pd.DataFrame(rows)
status

NameError: name 'fastas' is not defined

In [ ]:
status[["streme", "tomtom", "fimo_jaspar", "fimo_streme"]].sum()

In [ ]:
status[~(status["streme"] & status["tomtom"] & status["fimo_jaspar"] & status["fimo_streme"])]

## Import Results

### TOMTOM

In [ ]:
tomtom_tables = []

for fasta in fastas:
    p = paths_for_fasta(fasta)
    if exists_nonempty(p["tomtom_tsv"]):
        df = pd.read_csv(p["tomtom_tsv"], sep="\t", comment="#")
        df["dataset"] = p["name"]
        tomtom_tables.append(df)

tomtom_all = pd.concat(tomtom_tables, ignore_index=True) if tomtom_tables else pd.DataFrame()
tomtom_all.head()

In [ ]:
tomtom_all.sort_values(["dataset", "q-value"]).head(30)

### FIMO-JASPAR

In [ ]:
fimo_jaspar_tables = []

for fasta in fastas:
    p = paths_for_fasta(fasta)
    if exists_nonempty(p["fimo_jaspar_tsv"]):
        df = pd.read_csv(p["fimo_jaspar_tsv"], sep="\t", comment="#")
        df["dataset"] = p["name"]
        fimo_jaspar_tables.append(df)

fimo_jaspar_all = pd.concat(fimo_jaspar_tables, ignore_index=True) if fimo_jaspar_tables else pd.DataFrame()
fimo_jaspar_all.head()

In [ ]:
if not fimo_jaspar_all.empty:
    fimo_jaspar_all.groupby("dataset").size().sort_values(ascending=False)

### FIMO-STREME

In [ ]:
fimo_streme_tables = []

for fasta in fastas:
    p = paths_for_fasta(fasta)
    if exists_nonempty(p["fimo_streme_tsv"]):
        df = pd.read_csv(p["fimo_streme_tsv"], sep="\t", comment="#")
        df["dataset"] = p["name"]
        fimo_streme_tables.append(df)

fimo_streme_all = pd.concat(fimo_streme_tables, ignore_index=True) if fimo_streme_tables else pd.DataFrame()
fimo_streme_all.head()

## Save Results

In [ ]:
SUMMARY_DIR = RESULT_DIR / "summary_tables"
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

status.to_csv(SUMMARY_DIR / "meme_pipeline_status.csv", index=False)

if not tomtom_all.empty:
    tomtom_all.to_csv(SUMMARY_DIR / "tomtom_all.tsv", sep="\t", index=False)

if not fimo_jaspar_all.empty:
    fimo_jaspar_all.to_csv(SUMMARY_DIR / "fimo_jaspar_all.tsv", sep="\t", index=False)

if not fimo_streme_all.empty:
    fimo_streme_all.to_csv(SUMMARY_DIR / "fimo_streme_all.tsv", sep="\t", index=False)

print("Saved summary tables to:", SUMMARY_DIR)